# Draft + Free-Agency Scenario Model

This notebook connects possible Bulls draft outcomes to free-agent priorities. The idea is simple:

> If the Bulls draft one type of player, they should not use free agency to solve the exact same roster need unless the value is exceptional.

For example, if Chicago drafts a rim-protecting center, free agency should probably shift toward shooting, wing defense, or guard creation. If Chicago drafts a guard, free agency should probably emphasize size and frontcourt defense.

## Workflow

1. Load the draft prospect seed table.
2. Load and score the 2026 free-agent table.
3. Choose one prospect at pick 4 and one prospect at pick 15.
4. Adjust free-agent role priorities based on those draft picks.
5. Rank free agents for that specific offseason scenario.

In [1]:
from pathlib import Path
import re
import unicodedata

import pandas as pd

project_dir = Path.cwd()
if not (project_dir / "data").exists():
    project_dir = project_dir / "fresh_start"

data_dir = project_dir / "data"
prospect_path = data_dir / "draft_prospects_2026_seed.csv"
prospect_eval_path = data_dir / "prospect_evaluation_table_2026.csv"
free_agent_path = data_dir / "nba_free_agents_2026_spotrac.csv"
stats_path = data_dir / "nba_all_players_2526.csv"

prospects = pd.read_csv(prospect_eval_path if prospect_eval_path.exists() else prospect_path)
free_agents_raw = pd.read_csv(free_agent_path)
league_df = pd.read_csv(stats_path)

prospects.head()

,PLAYER_NAME,POSITION,SCHOOL_OR_TEAM,PROJECTED_PICK_MIN,PROJECTED_PICK_MAX,DRAFT_RANGE,PROSPECT_ROLE,CRAFTEDNBA_RANK,CRAFTEDNBA_SOURCE_URL,CRAFTEDNBA_SOURCE_NOTES,...,MAX_VERTICAL_LEAP,LANE_AGILITY_TIME,THREE_QUARTER_SPRINT,MEASUREMENT_SOURCE,MEASUREMENT_NOTES,COMBINE_SOURCE_URL,physical_score,college_production_score,prospect_eval_score,NOTES
0,Cameron Boozer,PF,Duke,1,4,top_4,frontcourt_star,3.0,https://craftednba.com/draft/prospects,Rank from CraftedNBA 2026 NBA Draft Prospect S...,...,35.0,11.06,3.31,nba_combine,NaN,https://www.nba.com/stats/draft/combine-anthro,0.280556,0.913681,0.597118,Elite production forward.
1,Caleb Wilson,F,North Carolina,3,6,bulls_pick_4,athletic_forward_defender,4.0,https://craftednba.com/draft/prospects,Rank from CraftedNBA 2026 NBA Draft Prospect S...,...,39.5,11.17,3.23,nba_combine,NaN,https://www.nba.com/stats/draft/combine-anthro,0.249496,0.485755,0.367625,Potential pick 4 frontcourt/wing option.
2,AJ Dybantsa,F,BYU,1,3,top_3,primary_wing_creator,2.0,https://craftednba.com/draft/prospects,Rank from CraftedNBA 2026 NBA Draft Prospect S...,...,42.0,11.06,3.14,NaN,NaN,https://www.nba.com/stats/draft/combine-anthro,0.407162,0.296829,0.351996,Likely unavailable at Bulls pick 4 unless he f...
3,Yaxel Lendeborg,F,Michigan,10,22,pick_15,older_two_way_forward,10.0,https://craftednba.com/draft/prospects,Rank from CraftedNBA 2026 NBA Draft Prospect S...,...,32.0,10.82,3.35,nba_combine,NaN,https://www.nba.com/stats/draft/combine-anthro,0.319176,0.168314,0.243745,Older forward; potentially more NBA-ready.
4,Cameron Carr,G/F,Baylor,10,22,pick_15,scoring_wing,21.0,https://craftednba.com/draft/prospects,Rank from CraftedNBA 2026 NBA Draft Prospect S...,...,42.5,10.46,3.17,nba_combine,NaN,https://www.nba.com/stats/draft/combine-anthro,0.454553,NaN,0.227276,Scoring wing.


## Prospect Data

The seed file is intentionally editable. Some stats are filled in, while others are blank until we add a more complete college/international data source. For now, the scenario logic depends most on `PROJECTED_PICK_MIN`, `PROJECTED_PICK_MAX`, and `PROSPECT_ROLE`.

In [2]:
draft_sort_columns = ["PROJECTED_PICK_MIN", "PROJECTED_PICK_MAX"]
if "CRAFTEDNBA_RANK" in prospects.columns:
    draft_sort_columns = ["CRAFTEDNBA_RANK", "PROJECTED_PICK_MIN", "PROJECTED_PICK_MAX"]

pick_4_pool = prospects[
    (prospects["PROJECTED_PICK_MIN"] <= 4)
    & (prospects["PROJECTED_PICK_MAX"] >= 4)
].sort_values(draft_sort_columns, na_position="last")

pick_15_pool = prospects[
    (prospects["PROJECTED_PICK_MIN"] <= 15)
    & (prospects["PROJECTED_PICK_MAX"] >= 15)
].sort_values(draft_sort_columns, na_position="last")

prospect_display_columns = [
    "PLAYER_NAME", "POSITION", "SCHOOL_OR_TEAM", "PROJECTED_PICK_MIN", "PROJECTED_PICK_MAX", "PROSPECT_ROLE"
]
if "CRAFTEDNBA_RANK" in prospects.columns:
    prospect_display_columns.insert(3, "CRAFTEDNBA_RANK")
if "prospect_eval_score" in prospects.columns:
    prospect_display_columns.append("prospect_eval_score")

print("Potential pick 4 pool:")
display(pick_4_pool[prospect_display_columns])

print("Potential pick 15 pool:")
display(pick_15_pool[prospect_display_columns])

Potential pick 4 pool:


,PLAYER_NAME,POSITION,SCHOOL_OR_TEAM,CRAFTEDNBA_RANK,PROJECTED_PICK_MIN,PROJECTED_PICK_MAX,PROSPECT_ROLE,prospect_eval_score
11,Darryn Peterson,G,Kansas,1.0,1,4,lead_guard_creator,-0.175540
0,Cameron Boozer,PF,Duke,3.0,1,4,frontcourt_star,0.597118
1,Caleb Wilson,F,North Carolina,4.0,3,6,athletic_forward_defender,0.367625
12,Koa Peat,F,Arizona,8.0,4,8,two_way_forward,-0.200760
7,Chris Cenac Jr.,F/C,Houston,18.0,4,10,rim_running_big,0.017949


Potential pick 15 pool:


,PLAYER_NAME,POSITION,SCHOOL_OR_TEAM,CRAFTEDNBA_RANK,PROJECTED_PICK_MIN,PROJECTED_PICK_MAX,PROSPECT_ROLE,prospect_eval_score
5,Nate Ament,F,Tennessee,9.0,8,18,stretch_forward,0.053758
3,Yaxel Lendeborg,F,Michigan,10.0,10,22,older_two_way_forward,0.243745
16,Keaton Wagler,G,Illinois,15.0,8,18,shooting_guard,-0.406832
14,Darius Acuff Jr.,G,Arkansas,17.0,6,16,scoring_guard,-0.201289
4,Cameron Carr,G/F,Baylor,21.0,10,22,scoring_wing,0.227276
6,Aday Mara,C,Michigan,29.0,8,18,rim_protecting_big,0.050866
17,Meleek Thomas,G,Arkansas,32.0,12,24,scoring_guard,-0.421922
15,Karim Lopez,F,New Zealand Breakers,NaN,7,15,international_two_way_forward,-0.202327
10,Allen Graves,F,Santa Clara,NaN,15,30,stretch_forward_defender,-0.160964


## Rebuild The Free-Agent Decision Board

This recreates the same core free-agent fit model from `free_agent_fit_2026.ipynb`, so this notebook can run independently.

In [3]:
def clean_money(value):
    if pd.isna(value):
        return pd.NA
    cleaned = re.sub(r"[^0-9.]", "", str(value))
    return float(cleaned) if cleaned else pd.NA

def normalize_player_name(name):
    if pd.isna(name):
        return ""
    name = str(name).lower()
    name = unicodedata.normalize("NFKD", name)
    name = "".join(char for char in name if not unicodedata.combining(char))
    for char in [".", "'", "’", "‘", "`"]:
        name = name.replace(char, "")
    name = re.sub(r"[^a-z0-9 ]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    name = re.sub(r"\b(jr|sr|ii|iii|iv)\b", "", name)
    return re.sub(r"\s+", " ", name).strip()

name_aliases = {
    "nahshon hyland": "bones hyland",
    "cameron thomas": "cam thomas",
}

def assign_role_label(row):
    pos = str(row["POS"])
    if pos == "C" and row["BLK"] >= 1.0 and row["REB"] >= 6.0:
        return "rim-protecting big"
    if pos in ["PF", "C"] and row["REB"] >= 6.0:
        return "frontcourt rebounder"
    if pos in ["SF", "PF", "SG"] and row["STL"] >= 0.9 and row["REB"] >= 4.0:
        return "wing defender"
    if row["FG3_PCT"] >= 0.37 and row["FG3M"] >= 1.5:
        return "floor spacer"
    if pos in ["PG", "SG", "G"] and row["AST"] >= 3.0:
        return "guard creator"
    if row["PTS"] >= 12.0:
        return "secondary scorer"
    return "depth option"

def assign_target_tier(row):
    salary = row["PREV_AAV_CLEAN"]
    fit = row["fit_score"]
    fa_class = row["FA_CLASS"]
    if fa_class == "UFA" and fit >= 0.75 and salary <= 15_000_000:
        return "priority realistic target"
    if fa_class == "UFA" and fit >= 0.75 and salary <= 30_000_000:
        return "priority expensive target"
    if fa_class == "RFA" and fit >= 1.00:
        return "high fit but difficult"
    if salary > 25_000_000:
        return "expensive / low flexibility"
    if fit >= 0.40:
        return "secondary target"
    return "low-priority depth"

In [4]:
free_agents = free_agents_raw.copy()
free_agents = free_agents.rename(columns={free_agents.columns[0]: "PLAYER_NAME"})
free_agents.columns = [
    str(col).strip().upper().replace(" ", "_").replace("(", "").replace(")", "")
    for col in free_agents.columns
]
free_agents["PREV_AAV_CLEAN"] = free_agents["PREV_AAV"].apply(clean_money)
free_agents["FA_CLASS"] = free_agents["TYPE"].str.split("/").str[0].str.strip()
free_agents["RIGHTS_OR_OPTION"] = free_agents["TYPE"].str.split("/").str[1].str.strip()
free_agents["IS_DEFAULT_AVAILABLE"] = free_agents["FA_CLASS"].isin(["UFA", "RFA"])
free_agents["NAME_KEY"] = free_agents["PLAYER_NAME"].apply(normalize_player_name).replace(name_aliases)

league_stats = league_df.copy()
league_stats["NAME_KEY"] = league_stats["PLAYER_NAME"].apply(normalize_player_name)

fa_stats = free_agents.merge(
    league_stats,
    on="NAME_KEY",
    how="left",
    suffixes=("_FA", "_STATS"),
)

score_features = {
    "BLK": 0.22,
    "DREB": 0.18,
    "REB": 0.12,
    "STL": 0.12,
    "FG3_PCT": 0.10,
    "FG3M": 0.08,
    "PTS": 0.08,
    "PLUS_MINUS": 0.06,
    "AGE_STATS": -0.04,
}

scored = fa_stats[fa_stats["PLAYER_ID"].notna()].copy()
for feature, weight in score_features.items():
    mean = scored[feature].mean()
    std = scored[feature].std(ddof=0)
    scored[f"{feature}_z"] = (scored[feature] - mean) / std if std else 0

scored["fit_score"] = sum(
    scored[f"{feature}_z"] * weight
    for feature, weight in score_features.items()
)

available_fa = scored[
    scored["IS_DEFAULT_AVAILABLE"]
    & (scored["TEAM_ABBREVIATION"] != "CHI")
    & (scored["GP"] >= 20)
    & (scored["PREV_AAV_CLEAN"] <= 35_000_000)
].copy()

available_fa["role_label"] = available_fa.apply(assign_role_label, axis=1)
available_fa["target_tier"] = available_fa.apply(assign_target_tier, axis=1)

available_fa[["PLAYER_NAME_FA", "POS", "AGE_FA", "FA_CLASS", "PREV_AAV_CLEAN", "fit_score", "role_label", "target_tier"]].head()

,PLAYER_NAME_FA,POS,AGE_FA,FA_CLASS,PREV_AAV_CLEAN,fit_score,role_label,target_tier
4,C.J. McCollum,SG,34.7,UFA,32000000.0,0.664019,floor spacer,expensive / low flexibility
5,Khris Middleton,SF,34.8,UFA,31000000.0,-0.059659,depth option,expensive / low flexibility
6,Kristaps Porzingis,PF,30.8,UFA,30000000.0,1.282666,secondary scorer,priority expensive target
9,Tobias Harris,PF,33.8,UFA,26000000.0,0.818541,wing defender,priority expensive target
13,John Collins,PF,28.7,UFA,25000000.0,0.928853,wing defender,priority expensive target


## Scenario Logic

Each draft prospect has a `PROSPECT_ROLE`. That role tells the model which free-agent roles become less urgent and which complementary roles become more urgent.

This is not meant to replace scouting. It is a way to test roster-building logic.

In [5]:
base_role_priority = {
    "rim-protecting big": 1.10,
    "frontcourt rebounder": 1.00,
    "wing defender": 0.90,
    "floor spacer": 0.80,
    "guard creator": 0.70,
    "secondary scorer": 0.65,
    "depth option": 0.35,
}

draft_role_effects = {
    "rim_protecting_big": {
        "covered": ["rim-protecting big", "frontcourt rebounder"],
        "boosted": ["floor spacer", "wing defender", "guard creator"],
    },
    "rim_running_big": {
        "covered": ["rim-protecting big", "frontcourt rebounder"],
        "boosted": ["floor spacer", "wing defender", "guard creator"],
    },
    "frontcourt_star": {
        "covered": ["frontcourt rebounder", "secondary scorer"],
        "boosted": ["rim-protecting big", "floor spacer", "wing defender"],
    },
    "two_way_forward": {
        "covered": ["wing defender", "secondary scorer"],
        "boosted": ["rim-protecting big", "guard creator", "floor spacer"],
    },
    "athletic_forward_defender": {
        "covered": ["wing defender", "frontcourt rebounder"],
        "boosted": ["rim-protecting big", "guard creator", "floor spacer"],
    },
    "primary_wing_creator": {
        "covered": ["secondary scorer", "wing defender"],
        "boosted": ["rim-protecting big", "frontcourt rebounder", "floor spacer"],
    },
    "lead_guard_creator": {
        "covered": ["guard creator", "secondary scorer"],
        "boosted": ["rim-protecting big", "frontcourt rebounder", "wing defender"],
    },
    "scoring_guard": {
        "covered": ["guard creator", "secondary scorer"],
        "boosted": ["rim-protecting big", "wing defender", "frontcourt rebounder"],
    },
    "shooting_guard": {
        "covered": ["floor spacer", "secondary scorer"],
        "boosted": ["rim-protecting big", "wing defender", "frontcourt rebounder"],
    },
    "scoring_wing": {
        "covered": ["secondary scorer", "floor spacer"],
        "boosted": ["rim-protecting big", "frontcourt rebounder", "guard creator"],
    },
    "stretch_forward": {
        "covered": ["floor spacer", "frontcourt rebounder"],
        "boosted": ["rim-protecting big", "guard creator", "wing defender"],
    },
    "stretch_forward_defender": {
        "covered": ["floor spacer", "wing defender"],
        "boosted": ["rim-protecting big", "guard creator", "frontcourt rebounder"],
    },
    "older_two_way_forward": {
        "covered": ["wing defender", "frontcourt rebounder"],
        "boosted": ["rim-protecting big", "guard creator", "floor spacer"],
    },
    "international_two_way_forward": {
        "covered": ["wing defender", "secondary scorer"],
        "boosted": ["rim-protecting big", "guard creator", "frontcourt rebounder"],
    },
}

def adjusted_role_priorities(drafted_roles):
    priorities = base_role_priority.copy()
    for draft_role in drafted_roles:
        effect = draft_role_effects.get(draft_role, {"covered": [], "boosted": []})
        for role in effect["covered"]:
            priorities[role] = max(0.25, priorities.get(role, 0.5) - 0.35)
        for role in effect["boosted"]:
            priorities[role] = min(1.35, priorities.get(role, 0.5) + 0.15)
    return priorities

def score_free_agents_for_draft_scenario(pick_4_name, pick_15_name, top_n=20):
    drafted = prospects[prospects["PLAYER_NAME"].isin([pick_4_name, pick_15_name])].copy()
    drafted_roles = drafted["PROSPECT_ROLE"].tolist()
    role_priorities = adjusted_role_priorities(drafted_roles)

    board = available_fa.copy()
    board["scenario_role_priority"] = board["role_label"].map(role_priorities).fillna(0.50)
    board["salary_penalty"] = board["PREV_AAV_CLEAN"].fillna(0) / 100_000_000
    board["scenario_score"] = (
        board["fit_score"] * board["scenario_role_priority"]
        - board["salary_penalty"]
    )

    return (
        board.sort_values("scenario_score", ascending=False)
        [[
            "PLAYER_NAME_FA", "POS", "AGE_FA", "FA_CLASS", "PREV_TEAM",
            "PREV_AAV_CLEAN", "fit_score", "role_label", "target_tier",
            "scenario_role_priority", "scenario_score"
        ]]
        .head(top_n)
        .reset_index(drop=True)
    )

## Try One Scenario

Change the names below to test different draft outcomes. The names must match the `PLAYER_NAME` values in the prospect table.

In [6]:
pick_4_choice = "Cameron Boozer"
pick_15_choice = "Aday Mara"

score_free_agents_for_draft_scenario(pick_4_choice, pick_15_choice, top_n=20)

,PLAYER_NAME_FA,POS,AGE_FA,FA_CLASS,PREV_TEAM,PREV_AAV_CLEAN,fit_score,role_label,target_tier,scenario_role_priority,scenario_score
0,Peyton Watson,SG,23.8,RFA,DEN,2816879.0,1.400599,wing defender,high fit but difficult,1.20,1.652550
1,Robert Williams III,C,28.6,UFA,POR,12000000.0,1.324957,rim-protecting big,priority realistic target,0.90,1.072462
2,Kelly Oubre Jr.,SF,30.5,UFA,PHI,8182575.0,0.922304,wing defender,priority realistic target,1.20,1.024939
3,Mitchell Robinson,C,28.2,UFA,NYK,15000000.0,1.147332,rim-protecting big,priority realistic target,0.90,0.882599
4,John Collins,PF,28.7,UFA,LAC,25000000.0,0.928853,wing defender,priority expensive target,1.20,0.864624
5,Collin Gillespie,PG,26.9,UFA,PHX,2378870.0,0.773862,floor spacer,priority realistic target,1.10,0.827459
6,Tobias Harris,PF,33.8,UFA,DET,26000000.0,0.818541,wing defender,priority expensive target,1.20,0.722250
7,Jordan Goodwin,PG,27.6,UFA,PHX,1286648.0,0.598362,floor spacer,secondary target,1.10,0.645331
8,Norman Powell,SG,33.0,UFA,MIA,18000000.0,0.708726,floor spacer,secondary target,1.10,0.599599
9,Ayo Dosunmu,PG,26.3,UFA,MIN,7000000.0,0.516070,floor spacer,secondary target,1.10,0.497677


## Build Scenario Grid

This creates every pick 4 / pick 15 combination from the projected availability pools and records the top free-agent target for each scenario. It gives you a fast way to see how free agency changes when the draft changes.

In [7]:
scenario_rows = []

for pick_4_name in pick_4_pool["PLAYER_NAME"]:
    for pick_15_name in pick_15_pool["PLAYER_NAME"]:
        scenario_targets = score_free_agents_for_draft_scenario(pick_4_name, pick_15_name, top_n=5)
        top_target = scenario_targets.iloc[0]
        scenario_rows.append({
            "pick_4": pick_4_name,
            "pick_15": pick_15_name,
            "top_fa_target": top_target["PLAYER_NAME_FA"],
            "top_fa_role": top_target["role_label"],
            "top_fa_tier": top_target["target_tier"],
            "top_fa_prev_aav": top_target["PREV_AAV_CLEAN"],
            "top_fa_scenario_score": top_target["scenario_score"],
        })

scenario_grid = pd.DataFrame(scenario_rows).sort_values(
    "top_fa_scenario_score",
    ascending=False,
).reset_index(drop=True)

scenario_grid.head(30)

,pick_4,pick_15,top_fa_target,top_fa_role,top_fa_tier,top_fa_prev_aav,top_fa_scenario_score
0,Darryn Peterson,Keaton Wagler,Jalen Duren,frontcourt rebounder,high fit but difficult,4868736.0,2.007067
1,Darryn Peterson,Darius Acuff Jr.,Jalen Duren,frontcourt rebounder,high fit but difficult,4868736.0,2.007067
2,Darryn Peterson,Cameron Carr,Jalen Duren,frontcourt rebounder,high fit but difficult,4868736.0,2.007067
3,Darryn Peterson,Meleek Thomas,Jalen Duren,frontcourt rebounder,high fit but difficult,4868736.0,2.007067
4,Darryn Peterson,Karim Lopez,Jalen Duren,frontcourt rebounder,high fit but difficult,4868736.0,2.007067
5,Darryn Peterson,Allen Graves,Jalen Duren,frontcourt rebounder,high fit but difficult,4868736.0,2.007067
6,Koa Peat,Meleek Thomas,Jalen Duren,frontcourt rebounder,high fit but difficult,4868736.0,1.769865
7,Koa Peat,Allen Graves,Jalen Duren,frontcourt rebounder,high fit but difficult,4868736.0,1.769865
8,Koa Peat,Karim Lopez,Jalen Duren,frontcourt rebounder,high fit but difficult,4868736.0,1.769865
9,Koa Peat,Cameron Carr,Jalen Duren,frontcourt rebounder,high fit but difficult,4868736.0,1.769865


## Next Steps

- Fill in missing prospect stats in `draft_prospects_2026_seed.csv`.
- Add a draft prospect score using normalized college stats.
- Add projected rookie salary slots for picks 4 and 15.
- Add cap-space math so each draft/free-agent scenario shows remaining spending room.
- Add manual scouting notes and injury risk flags.